<h1><strong><center> Final Model

После тюнинга гиперпараметров моделей с помощью Optuna, мы получили улучшение результатов всех моделей, кроме ExtraTreesClassifier. Поэтому в итог возьмем все базовые модели после Optuna, кроме ExtraTreesClassifier. Ее оставим с дефолтными гиперпараметрами.

In [18]:
import pandas as pd
import pickle
from catboost import CatBoostClassifier
import xgboost as xgb
from sklearn.base import BaseEstimator, ClassifierMixin
import warnings

warnings.filterwarnings('error')
warnings.filterwarnings('ignore')

In [19]:
'''
Обертка для XGBClassifier, добавляющая метод __sklearn_tags__  
для совместимости со scikit-learn 1.7 и выше.  
Решает проблему с DeprecationWarning
'''


class SklearnXGBClassifier(xgb.XGBClassifier, BaseEstimator, ClassifierMixin):
    def __sklearn_tags__(self):
        return {
            'non_deterministic': True,
            'requires_fit': True,
            'X_types': ['2darray'],
        }

In [20]:
file_path = '../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

In [21]:
# Загрузка моделей

model_path = '../../models/2_optuna_models/optuna_catboost_model.cb'
catboost_model = CatBoostClassifier()
catboost_model.load_model(model_path)

with open('../../models/2_optuna_models/optuna_gradient_boosting_model.pkl', 'rb') as file:
    gradient_boosting_model = pickle.load(file)

with open('../../models/2_optuna_models/optuna_logistic_regression_model.pkl', 'rb') as file:
    logistic_regression_model = pickle.load(file)

with open('../../models/2_optuna_models/optuna_random_forest_model.pkl', 'rb') as file:
    random_forest_model = pickle.load(file)

with open('../../models/1_default_models/et_model.pkl', 'rb') as file:
    extra_trees_model = pickle.load(file)

In [22]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score

# Инициализация стратифицированного KFold
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Списки для сбора предсказаний и меток
all_meta_preds = []
all_true_labels = []

# Внешний цикл кросс-валидации
for outer_train_idx, outer_test_idx in outer_cv.split(X, y):
    X_outer_train, X_outer_test = X.iloc[outer_train_idx], X.iloc[outer_test_idx]
    y_outer_train, y_outer_test = y.iloc[outer_train_idx], y.iloc[outer_test_idx]
    
    # Списки для out-of-fold предсказаний
    catboost_preds, lr_preds, rf_preds, et_preds, gb_preds, meta_labels = [], [], [], [], [], []
    
    # Внутренний цикл для генерации мета-признаков
    for inner_train_idx, inner_val_idx in inner_cv.split(X_outer_train, y_outer_train):
        X_inner_train = X_outer_train.iloc[inner_train_idx]
        y_inner_train = y_outer_train.iloc[inner_train_idx]
        X_inner_val = X_outer_train.iloc[inner_val_idx]
        y_inner_val = y_outer_train.iloc[inner_val_idx]
        
        # Инициализация моделей с оптимальными параметрами
        models = {
            'catboost': clone(catboost_model),
            'lr': clone(logistic_regression_model),
            'rf': clone(random_forest_model),
            'et': clone(extra_trees_model),
            'gb': clone(gradient_boosting_model)
        }
        
        # Обучение и предсказание для каждой модели
        fold_preds = {}
        for name, model in models.items():
            if name == 'catboost':
                model.fit(X_inner_train, y_inner_train, verbose=0)
            else:
                model.fit(X_inner_train, y_inner_train)
            pred = model.predict(X_inner_val)
            # Специальная обработка CatBoost
            if name == 'catboost':
                pred = pred.ravel()
            fold_preds[name] = pred
        
        # Сохранение предсказаний
        catboost_preds.append(fold_preds['catboost'])
        lr_preds.append(fold_preds['lr'])
        rf_preds.append(fold_preds['rf'])
        et_preds.append(fold_preds['et'])
        gb_preds.append(fold_preds['gb'])
        meta_labels.append(y_inner_val)
    
    # Создание мета-признаков
    X_meta_train = pd.DataFrame({
        'catboost': np.concatenate(catboost_preds),
        'lr': np.concatenate(lr_preds),
        'rf': np.concatenate(rf_preds),
        'et': np.concatenate(et_preds),
        'gb': np.concatenate(gb_preds)
    })
    
    y_meta_train = np.concatenate(meta_labels)
    
    # Обучение мета-модели
    meta_model = xgb.XGBClassifier(random_state=42, tree_method="auto")
    meta_model.fit(X_meta_train, y_meta_train)
    
    # Обучение финальных моделей на полном наборе данных
    final_models = {
        'catboost': clone(catboost_model).fit(X_outer_train, y_outer_train),
        'lr': clone(logistic_regression_model).fit(X_outer_train, y_outer_train),
        'rf': clone(random_forest_model).fit(X_outer_train, y_outer_train),
        'et': clone(extra_trees_model).fit(X_outer_train, y_outer_train),
        'gb': clone(gradient_boosting_model).fit(X_outer_train, y_outer_train)
    }
    
    # Генерация тестовых предсказаний
    X_meta_test = pd.DataFrame({
        'catboost': final_models['catboost'].predict(X_outer_test).ravel(),
        'lr': final_models['lr'].predict(X_outer_test).ravel(),
        'rf': final_models['rf'].predict(X_outer_test).ravel(),
        'et': final_models['et'].predict(X_outer_test).ravel(),
        'gb': final_models['gb'].predict(X_outer_test).ravel()
    })
    
    # Предсказание и сохранение результатов
    final_preds = meta_model.predict(X_meta_test)
    all_meta_preds.extend(final_preds)
    all_true_labels.extend(y_outer_test)

# Оценка качества
balanced_acc = balanced_accuracy_score(all_true_labels, all_meta_preds)
print(f'Stratified Cross-Validation Balanced Accuracy: {balanced_acc:.4f}')

Stratified Cross-Validation Balanced Accuracy: 0.9942


Видим повышение точности по сравнению с блендинговых базовых моделей с дефолтными параметрами. Теперь сохраним модели и мета-модель.

In [23]:
with open('../../models/3_final_models/optuna_gradient_boosting_model.pkl', 'wb') as file:
    pickle.dump(gradient_boosting_model, file)

with open('../../models/3_final_models/optuna_logistic_regression_model.pkl', 'wb') as file:
    pickle.dump(logistic_regression_model, file)
    
with open('../../models/3_final_models/optuna_random_forest_model.pkl', 'wb') as file:
    pickle.dump(random_forest_model, file)
    
with open('../../models/3_final_models/et_model.pkl', 'wb') as file:
    pickle.dump(extra_trees_model, file)
    
with open('../../models/3_final_models/meta_model_xgboost.pkl', 'wb') as file:
    pickle.dump(meta_model, file)

catboost_model.save_model('../../models/3_final_models/catboost_model.cb')